## 1. Setup

In [ ]:
import sys
sys.path.append("..")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import yfinance as yf

from src.paths import MANUAL_PATH, PROCESSED_PATH
from src.risk import (
    annualized_return,
    annualized_volatility,
    sharpe_ratio,
    max_drawdown,
    calmar_ratio,
    drawdown_series,
    herfindahl_index,
    effective_number_of_positions as compute_effective_number_of_positions,
    top_n_weight,
    compute_risk_contribution,
)

## 2. Paths & Load Data

In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROCESSED_PATH

holdings = pd.read_csv(DATA_PATH / "current_holdings_trackable.csv")
actual_vs_target = pd.read_csv(DATA_PATH / "actual_vs_target_weights.csv")
portfolio_kpis = pd.read_csv(DATA_PATH / "portfolio_kpis.csv")

holdings.head()

In [ ]:
holdings.columns.tolist()

In [ ]:
actual_vs_target.columns.tolist()

## 3. Check Relevant Columns

In [ ]:
required_holdings_cols = [
    "symbol",
    "name",
    "asset_class",
    "yf_ticker",
    "price_currency",
    "market_value",
    "actual_weight"
]

missing_holdings_cols = [
    col for col in required_holdings_cols
    if col not in holdings.columns
]

assert len(missing_holdings_cols) == 0

In [ ]:
required_target_cols = [
    "symbol",
    "target_weight"
]

missing_target_cols = [
    col for col in required_target_cols
    if col not in actual_vs_target.columns
]

assert len(missing_target_cols) == 0

## 4. Prepare Tickers

In [ ]:
tickers = (
    holdings["yf_ticker"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

# Optional history_proxy per ticker (same index/underlying, longer price
# history), maintained in the asset universe config of notebook 04. Used in
# section 8 to back-fill returns before an asset's own data starts, so the
# youngest listing (e.g. an ETF share class launched in 2024) doesn't cut the
# whole backtest window down to a crash-free bull market.
UNIVERSE_PATH = MANUAL_PATH / "asset_universe_manual.csv"

history_proxy_map = {}

if UNIVERSE_PATH.exists():
    universe = pd.read_csv(UNIVERSE_PATH)
    if "history_proxy" in universe.columns:
        history_proxy_map = (
            universe
            .dropna(subset=["history_proxy"])
            .set_index("yf_ticker")["history_proxy"]
            .reindex(tickers)
            .dropna()
            .to_dict()
        )

proxy_tickers = [
    proxy for proxy in dict.fromkeys(history_proxy_map.values())
    if proxy not in tickers
]

print("History proxies:", history_proxy_map or "none")
tickers

In [ ]:
ticker_currency = (
    holdings
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")["price_currency"]
    .to_dict()
)

# Proxies carry no price currency of their own here; only EUR listings are
# supported so no extra FX handling is needed.
for proxy in proxy_tickers:
    assert proxy.endswith((".DE", "-EUR")), f"history proxy {proxy} must be EUR-listed"
    ticker_currency[proxy] = "EUR"

ticker_currency

## 5. Download historical prices

In [ ]:
START_DATE = "2020-01-01"
END_DATE = None

In [ ]:
price_data = {}

for ticker in tickers + proxy_tickers:
    hist = yf.download(
        ticker,
        start=START_DATE,
        end=END_DATE,
        auto_adjust=True,
        progress=False
    )

    if hist.empty:
        print(f"No data for {ticker}")
        continue

    close = hist["Close"]

    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]

    price_data[ticker] = close.rename(ticker)

prices_raw = pd.concat(price_data.values(), axis=1)

prices_raw.head()

In [ ]:
prices_raw.isna().mean().sort_values(ascending=False)

## 6: FX: USD/EUR

In [ ]:
eurusd = yf.download(
    "EURUSD=X",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=True,
    progress=False
)

eurusd_close = eurusd["Close"]

if isinstance(eurusd_close, pd.DataFrame):
    eurusd_close = eurusd_close.iloc[:, 0]

usd_to_eur = 1 / eurusd_close
usd_to_eur.name = "USD_TO_EUR"

usd_to_eur.tail()

## 7. Convert prices to EUR

In [ ]:
prices_eur = prices_raw.copy()

for ticker in prices_eur.columns:
    if ticker_currency.get(ticker) == "USD":
        prices_eur[ticker] = prices_eur[ticker] * usd_to_eur

prices_eur = prices_eur.dropna(how="all")

# Drop today's row: during (or right after) the trading day Yahoo returns an
# incomplete bar — intraday prices for open markets, no close yet for others
# (forward-filled into a fake 0% day). Every analysis here works on completed
# sessions only.
prices_eur = prices_eur.loc[prices_eur.index < pd.Timestamp.today().normalize()]

prices_eur.tail()

In [ ]:
prices_eur.isna().mean().sort_values(ascending=False)

## 8. Compute returns

In [ ]:
non_crypto_tickers = [
    ticker for ticker in prices_eur.columns
    if ticker != "BTC-EUR"
]

trading_days = prices_eur[non_crypto_tickers].notna().any(axis=1)

prices_eur_trading = prices_eur.loc[trading_days].copy()

# Yahoo often leaves the latest European close empty (volume, but no price)
# for hours after the session. Forward-filling that gap would fake a 0% day
# for those assets and push the real move into the next day, so stop at the
# last date on which every exchange-traded holding has a real close.
exchange_holdings = [
    ticker for ticker in tickers
    if ticker != "BTC-EUR" and ticker in prices_eur_trading.columns
]
last_complete_date = (
    prices_eur_trading[exchange_holdings]
    .apply(lambda x: x.last_valid_index())
    .min()
)
if last_complete_date < prices_eur_trading.index.max():
    print(f"Latest close missing for some holdings — data cut at {last_complete_date.date()}")
prices_eur_trading = prices_eur_trading.loc[:last_complete_date]

# Stocks/ETFs: forward-fill missing prices within trading days
prices_eur_trading[non_crypto_tickers] = (
    prices_eur_trading[non_crypto_tickers]
    .ffill()
)

# BTC: align onto the same portfolio trading days
if "BTC-EUR" in prices_eur_trading.columns:
    prices_eur_trading["BTC-EUR"] = (
        prices_eur["BTC-EUR"]
        .reindex(prices_eur_trading.index)
        .ffill()
    )

returns = prices_eur_trading.pct_change(fill_method=None)

# Back-fill each holding's returns with its history proxy before its own
# returns start (same approach as notebook 04, section 9).
for ticker, proxy in history_proxy_map.items():
    if ticker not in returns.columns or proxy not in returns.columns:
        continue

    own_start = returns[ticker].first_valid_index()
    before_own_start = returns.index < own_start

    returns.loc[before_own_start, ticker] = returns.loc[before_own_start, proxy]

returns = returns[[ticker for ticker in tickers if ticker in returns.columns]]

# Common window: starts once every holding (incl. proxy history) has returns.
first_valid_dates = returns.apply(lambda x: x.first_valid_index())

common_start_date = first_valid_dates.max()

print(
    f"Backtest window: {common_start_date.date()} to {returns.index.max().date()} "
    f"(limited by {first_valid_dates.idxmax()})"
)

returns_filled = returns.loc[common_start_date:].fillna(0)

returns_filled.tail()

## 9. Actual Portfolio Weights

In [ ]:
actual_weights = (
    holdings
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")["actual_weight"]
)

actual_weights = actual_weights.reindex(returns_filled.columns).fillna(0)

assert np.isclose(actual_weights.sum(), 1.0)

## 10. Actual Portfolio Historical Performance

In [ ]:
actual_portfolio_returns = (
    returns_filled
    .mul(actual_weights, axis=1)
    .sum(axis=1)
)

actual_portfolio_value = (1 + actual_portfolio_returns).cumprod()

actual_portfolio_value.tail()

In [ ]:
actual_portfolio_cum_return_pct = (
    actual_portfolio_value - 1
) * 100

plt.figure(figsize=(12, 5))
plt.plot(
    actual_portfolio_cum_return_pct.index,
    actual_portfolio_cum_return_pct
)

plt.title("Backtested Performance of Current Portfolio Weights")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (%)")
plt.axhline(0, linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

## 11. Target Portfolio Weights

In [ ]:
target_weights = (
    actual_vs_target
    .merge(
        holdings[["symbol", "yf_ticker"]],
        on="symbol",
        how="left"
    )
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")["target_weight"]
)

target_weights = target_weights.reindex(returns_filled.columns).fillna(0)

assert np.isclose(target_weights.sum(), 1.0)

## 12. Target Portfolio Historical Performance

In [ ]:
target_portfolio_returns = (
    returns_filled
    .mul(target_weights, axis=1)
    .sum(axis=1)
)

target_portfolio_value = (1 + target_portfolio_returns).cumprod()

target_portfolio_cum_return_pct = (
    target_portfolio_value - 1
) * 100

target_portfolio_cum_return_pct.tail()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    actual_portfolio_cum_return_pct.index,
    actual_portfolio_cum_return_pct,
    label="Current Weights"
)

plt.plot(
    target_portfolio_cum_return_pct.index,
    target_portfolio_cum_return_pct,
    label="Target Weights"
)

plt.title("Backtested Performance: Current vs Target Portfolio Weights")
plt.xlabel("Date")
plt.ylabel("Cumulative Return (%)")
plt.axhline(0, linestyle="--", linewidth=1)
plt.legend()
plt.tight_layout()
plt.show()

## 13. Risk & Performance Metrics

In [ ]:
TRADING_DAYS = 252

# Risk-free rate (ECB deposit rate) is defined once in src/risk.py, see the
# rationale there — shared by notebooks 03, 04, 07 and 08.
from src.risk import RISK_FREE_RATE

In [ ]:
# annualized_return, annualized_volatility, sharpe_ratio, max_drawdown,
# calmar_ratio now imported from src.risk (see setup cell above).

In [ ]:
portfolio_metrics = pd.DataFrame({
    "Current Portfolio": {
        "Annualized Return": annualized_return(actual_portfolio_returns),
        "Annualized Volatility": annualized_volatility(actual_portfolio_returns),
        "Sharpe Ratio": sharpe_ratio(actual_portfolio_returns, RISK_FREE_RATE),
        "Max Drawdown": max_drawdown(actual_portfolio_returns),
        "Calmar Ratio": calmar_ratio(actual_portfolio_returns)
    },
    "Target Portfolio": {
        "Annualized Return": annualized_return(target_portfolio_returns),
        "Annualized Volatility": annualized_volatility(target_portfolio_returns),
        "Sharpe Ratio": sharpe_ratio(target_portfolio_returns, RISK_FREE_RATE),
        "Max Drawdown": max_drawdown(target_portfolio_returns),
        "Calmar Ratio": calmar_ratio(target_portfolio_returns)
    }
}).T

portfolio_metrics

In [ ]:
portfolio_metrics_display = portfolio_metrics.copy()

percentage_cols = [
    "Annualized Return",
    "Annualized Volatility",
    "Max Drawdown"
]

portfolio_metrics_display[percentage_cols] = (
    portfolio_metrics_display[percentage_cols] * 100
)

portfolio_metrics_display

## 14. Drawdown Analysis

In [ ]:
actual_drawdown = drawdown_series(actual_portfolio_returns)
target_drawdown = drawdown_series(target_portfolio_returns)

actual_drawdown.tail()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    actual_drawdown.index,
    actual_drawdown * 100,
    label="Current Weights"
)

plt.plot(
    target_drawdown.index,
    target_drawdown * 100,
    label="Target Weights"
)

plt.title("Portfolio Drawdowns")
plt.xlabel("Date")
plt.ylabel("Drawdown (%)")
plt.axhline(0, linestyle="--", linewidth=1)
plt.legend()
plt.tight_layout()
plt.show()

## 15. Rolling Volatility

In [ ]:
ROLLING_WINDOW = 63

rolling_vol_actual = (
    actual_portfolio_returns
    .rolling(ROLLING_WINDOW)
    .std()
    * np.sqrt(TRADING_DAYS)
)

rolling_vol_target = (
    target_portfolio_returns
    .rolling(ROLLING_WINDOW)
    .std()
    * np.sqrt(TRADING_DAYS)
)

rolling_vol_actual.tail()

In [ ]:
plt.figure(figsize=(12, 5))

plt.plot(
    rolling_vol_actual.index,
    rolling_vol_actual * 100,
    label="Current Weights"
)

plt.plot(
    rolling_vol_target.index,
    rolling_vol_target * 100,
    label="Target Weights"
)

plt.title("Rolling 3-Month Annualized Volatility")
plt.xlabel("Date")
plt.ylabel("Volatility (%)")
plt.legend()
plt.tight_layout()
plt.show()

## 16. Correlation Matrix

In [ ]:
asset_corr = returns_filled.corr()

asset_corr

In [ ]:
ticker_name_map = (
    holdings
    .dropna(subset=["yf_ticker"])
    .set_index("yf_ticker")["name"]
    .to_dict()
)

asset_corr_named = asset_corr.rename(
    index=ticker_name_map,
    columns=ticker_name_map
)

asset_corr_named

In [ ]:
plt.figure(figsize=(10, 8))

plt.imshow(asset_corr_named, aspect="auto")
plt.colorbar(label="Correlation")

plt.xticks(
    range(len(asset_corr_named.columns)),
    asset_corr_named.columns,
    rotation=90
)

plt.yticks(
    range(len(asset_corr_named.index)),
    asset_corr_named.index
)

plt.title("Asset Return Correlation Matrix")
plt.tight_layout()
plt.show()

## 17. Concentration Risk

In [ ]:
concentration = (
    holdings[
        [
            "symbol",
            "name",
            "asset_class",
            "market_value",
            "actual_weight"
        ]
    ]
    .sort_values("actual_weight", ascending=False)
    .copy()
)

concentration

In [ ]:
hhi = herfindahl_index(holdings["actual_weight"])

effective_number_of_positions = compute_effective_number_of_positions(holdings["actual_weight"])

hhi, effective_number_of_positions

In [ ]:
top_3_weight = top_n_weight(holdings["actual_weight"], 3)

top_5_weight = top_n_weight(holdings["actual_weight"], 5)

top_3_weight, top_5_weight

In [ ]:
concentration_metrics = pd.DataFrame({
    "metric": [
        "HHI",
        "Effective Number of Positions",
        "Top 3 Weight",
        "Top 5 Weight"
    ],
    "value": [
        hhi,
        effective_number_of_positions,
        top_3_weight,
        top_5_weight
    ]
})

concentration_metrics

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    concentration["name"],
    concentration["actual_weight"] * 100
)

plt.title("Portfolio Concentration by Position")
plt.xlabel("Position")
plt.ylabel("Portfolio Weight (%)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 18. Risk Contribution

In [ ]:
cov_matrix = returns_filled.cov() * TRADING_DAYS

weights = actual_weights.reindex(cov_matrix.columns).fillna(0)

portfolio_vol, risk_contribution, risk_contribution_pct = compute_risk_contribution(
    weights.values,
    cov_matrix.values
)

portfolio_vol

In [ ]:
risk_contribution_df = pd.DataFrame({
    "yf_ticker": cov_matrix.columns,
    "actual_weight": weights.values,
    "risk_contribution": risk_contribution,
    "risk_contribution_pct": risk_contribution_pct
})

risk_contribution_df = risk_contribution_df.merge(
    holdings[
        [
            "yf_ticker",
            "symbol",
            "name",
            "asset_class"
        ]
    ],
    on="yf_ticker",
    how="left"
)

risk_contribution_df = (
    risk_contribution_df
    .sort_values("risk_contribution_pct", ascending=False)
    .reset_index(drop=True)
)

risk_contribution_df

In [ ]:
plt.figure(figsize=(12, 5))

plt.bar(
    risk_contribution_df["name"],
    risk_contribution_df["risk_contribution_pct"] * 100
)

plt.title("Risk Contribution by Position")
plt.xlabel("Position")
plt.ylabel("Risk Contribution (%)")
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

## 19. Exports

In [ ]:
returns_filled.to_csv(
    DATA_PATH / "asset_returns_eur.csv"
)

portfolio_metrics.to_csv(
    DATA_PATH / "portfolio_risk_performance_metrics.csv"
)

portfolio_metrics_display.to_csv(
    DATA_PATH / "portfolio_risk_performance_metrics_display.csv"
)

concentration.to_csv(
    DATA_PATH / "portfolio_concentration.csv",
    index=False
)

concentration_metrics.to_csv(
    DATA_PATH / "portfolio_concentration_metrics.csv",
    index=False
)

risk_contribution_df.to_csv(
    DATA_PATH / "portfolio_risk_contribution.csv",
    index=False
)

In [ ]:
portfolio_risk_timeseries = pd.DataFrame({
    "actual_portfolio_return": actual_portfolio_returns,
    "target_portfolio_return": target_portfolio_returns,
    "actual_portfolio_value": actual_portfolio_value,
    "target_portfolio_value": target_portfolio_value,
    "actual_portfolio_cum_return_pct": actual_portfolio_cum_return_pct,
    "target_portfolio_cum_return_pct": target_portfolio_cum_return_pct,
    "actual_drawdown": actual_drawdown,
    "target_drawdown": target_drawdown,
    "rolling_vol_actual": rolling_vol_actual,
    "rolling_vol_target": rolling_vol_target
})

portfolio_risk_timeseries.to_csv(
    DATA_PATH / "portfolio_risk_timeseries.csv"
)

portfolio_risk_timeseries.tail()

## 20. Sanity Checks

In [ ]:
assert np.isclose(actual_weights.sum(), 1.0)
assert np.isclose(target_weights.sum(), 1.0)

assert not actual_portfolio_returns.isna().any()
assert not target_portfolio_returns.isna().any()

assert not portfolio_metrics.isna().all(axis=None)

assert np.isclose(
    risk_contribution_df["risk_contribution_pct"].sum(),
    1.0
)

print("All sanity checks passed.")